# 01 · Data exploration
Week 2 checkpoint: the corpus loads, one document can be displayed T0 → T3, and one similarity
metric is computed. Run `python -m src.data.preprocess` first.

In [1]:
import os, sys
ROOT = os.path.abspath("..")          # notebooks/ -> repository root
sys.path.insert(0, ROOT)
os.chdir(ROOT)
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## Raw corpus: 7 datasets × 2 files

In [2]:
from src.data.load_data import load_raw, load_raw_train
from src.utils.config import DOMAINS
rows = []
for d in DOMAINS:
    para, train = load_raw(d), load_raw_train(d)
    rows.append({"dataset": d, "domain": DOMAINS[d], "paraphrased_rows": len(para),
                 "train_rows": len(train), "sources": para.source.nunique(),
                 "versions": para.version_name.nunique()})
pd.DataFrame(rows)

,dataset,domain,paraphrased_rows,train_rows,sources,versions
0,xsum,news,70388,3243,7,22
1,yelp,review,72847,3344,7,22
2,cmv,opinion,35405,1666,7,22
3,eli5,qa,69270,3181,7,22
4,sci_gen,science,60166,2780,7,22
5,tldr,summary,53519,2559,7,22
6,wp,story,58417,2754,7,22


## Processed chains (after cleaning, alignment and filtering)

In [3]:
from src.data.load_data import load_chains, load_train
chains = load_chains()
pd.crosstab(chains.domain, chains.source, margins=True)

source,BigScience,Eleuther-AI,Human,LLAMA,OpenAI,PaLM,Tsinghua,All
domain,,,,,,,,
news,460,408,334,459,453,433,409,2956
opinion,224,181,227,237,233,159,228,1489
qa,440,359,434,444,457,420,449,3003
review,447,434,429,473,445,410,459,3097
science,177,98,227,233,465,231,203,1634
story,318,220,255,311,416,87,311,1918
summary,344,291,363,348,373,234,336,2289
All,2410,1991,2269,2505,2842,1974,2395,16386


## One document, T0 → T3

In [4]:
from src.data.preprocess import show_example
show_example(chains, "xsum-3981", paraphraser="chatgpt", source="Human", width=400)


Document xsum-3981 (news, source=Human) - paraphraser: chatgpt

[T0] (439 words)
Speaking to BBC Radio 4's Today programme, he said: "The only response can be:'yes, Mr President'." Gen Petraeus resigned as CIA director in 2012 following an extramarital affair with his biographer. It later emerged he had shared classified material with her. He served as a senior officer under Presidents Bush and Obama. He was the commander of US and Nato forces in Afghanistan, before retiring f...

[T1] (324 words)
During an interview with BBC Radio 4's Today programme, he stated that the appropriate response would be to say "yes, Mr President". General Petraeus resigned as the director of the CIA in 2012 due to an extramarital affair with his biographer, and it was later revealed that he had shared classified information with her. He served as a high-ranking officer under Presidents Bush and Obama, commandi...

[T2] (316 words)
In an interview with BBC Radio 4's Today programme, he commented that the 

## One similarity metric: SBERT cosine to T0 for that document

In [5]:
from src.similarity.sbert_similarity import compute_sbert_cosine
row = chains[(chains.key == "xsum-3981") & (chains.source == "Human")].iloc[0]
{g: round(compute_sbert_cosine([row[f"chatgpt_{g}"]], [row["T0"]]), 3) for g in ["T1", "T2", "T3"]}

[metrics] Loading SBERT model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'T1': 0.919, 'T2': 0.908, 'T3': 0.867}

## Training files for RQ2 (key-level train/val split)

In [6]:
train = load_train()
pd.crosstab(train.domain, train.split, margins=True)

split,train,val,All
domain,,,
news,2722,486,3208
opinion,1410,255,1665
qa,2619,472,3091
review,2720,483,3203
science,2358,416,2774
story,2332,416,2748
summary,2126,385,2511
All,16287,2913,19200
